### Invoice Lakebase & App

Provisions a Lakebase PostgreSQL instance and deploys the invoice terminal app:
- `procurement.sessions` — one row per browser session
- `procurement.conversations` — every chat turn (user message + agent response, latency, tokens)
- Deploys the `invoice-terminal` Databricks App that wraps the invoice supervisor endpoint

In [ ]:
%pip install databricks-sdk psycopg2-binary --upgrade

In [ ]:
import re
import time
import sys

sys.path.append('../utils')
from uc_state import add

CATALOG = dbutils.widgets.get("CATALOG")
INVOICE_SUPERVISOR_ENDPOINT_NAME = dbutils.widgets.get("INVOICE_SUPERVISOR_ENDPOINT_NAME")

INVOICE_LAKEBASE_INSTANCE_NAME = re.sub(r'[^a-z0-9-]', '-', f"{CATALOG}-invoice-supervisor".lower())
INVOICE_DB_NAME = "databricks_postgres"

print(f"Instance name: {INVOICE_LAKEBASE_INSTANCE_NAME}")
print(f"Supervisor endpoint: {INVOICE_SUPERVISOR_ENDPOINT_NAME}")

#### Lakebase Instance & Catalog

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import DatabaseInstance
from databricks.sdk.errors import NotFound, ResourceDoesNotExist

w = WorkspaceClient()
instance_name = INVOICE_LAKEBASE_INSTANCE_NAME

try:
    existing = w.database.get_database_instance(instance_name)
    print(f"Found existing database instance: {existing.name} (state: {existing.state})")
    instance = existing
except (NotFound, ResourceDoesNotExist):
    print(f"Creating new database instance: {instance_name}")
    instance = w.database.create_database_instance(
        DatabaseInstance(name=instance_name, capacity="CU_1")
    )
    print(f"Created database instance: {instance.name}")
    add(CATALOG, "databaseinstances", {"name": instance.name})

print("Waiting for database instance to be available...")
for _ in range(60):
    state = str(w.database.get_database_instance(instance_name).state)
    if state == "DatabaseInstanceState.AVAILABLE":
        print(f"Instance ready: {state}")
        break
    time.sleep(10)
else:
    raise TimeoutError(f"Database instance {instance_name} did not become available within 10 minutes")

In [ ]:
from databricks.sdk.service.database import DatabaseCatalog

catalog_name = INVOICE_LAKEBASE_INSTANCE_NAME

try:
    existing_cat = w.catalogs.get(catalog_name)
    print(f"Found existing catalog: {existing_cat.name}")
except Exception:
    catalog = w.database.create_database_catalog(
        DatabaseCatalog(
            name=catalog_name,
            database_instance_name=INVOICE_LAKEBASE_INSTANCE_NAME,
            database_name=INVOICE_DB_NAME,
            create_database_if_not_exists=True
        )
    )
    print(f"Created new database and catalog: {catalog.name}")
    add(CATALOG, "databasecatalogs", catalog)

#### OLTP Tables (Postgres)

In [ ]:
import psycopg2

PROJECT = f"projects/{INVOICE_LAKEBASE_INSTANCE_NAME}"
branches = list(w.postgres.list_branches(PROJECT))
default_branch = next(
    (b for b in branches if getattr(getattr(b, "status", None), "default", False)),
    branches[0]
)
branch_name = default_branch.name

endpoints = list(w.postgres.list_endpoints(parent=branch_name))
if not endpoints:
    for _ in range(12):
        time.sleep(5)
        endpoints = list(w.postgres.list_endpoints(parent=branch_name))
        if endpoints:
            break
    if not endpoints:
        raise RuntimeError("No Lakebase endpoint found")

endpoint = next(
    (e for e in endpoints if str(getattr(getattr(e, "status", None), "endpoint_type", "")).endswith("READ_WRITE")),
    endpoints[0]
)
endpoint_name = endpoint.name
endpoint_host = endpoint.status.hosts.host

print(f"Branch: {branch_name}")
print(f"Endpoint: {endpoint_name}")
print(f"Host: {endpoint_host}")

creds = w.postgres.generate_database_credential(endpoint=endpoint_name)
current_user = w.current_user.me().user_name

conn = psycopg2.connect(
    host=endpoint_host,
    port=5432,
    dbname=INVOICE_DB_NAME,
    user=current_user,
    password=creds.token,
    sslmode="require",
)
conn.autocommit = False

with conn.cursor() as cur:
    cur.execute("CREATE SCHEMA IF NOT EXISTS procurement")

    cur.execute("""
        CREATE TABLE IF NOT EXISTS procurement.sessions (
          session_id TEXT PRIMARY KEY,
          started_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
          user_agent TEXT
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS procurement.conversations (
          turn_id BIGSERIAL PRIMARY KEY,
          session_id TEXT NOT NULL REFERENCES procurement.sessions(session_id),
          turn_index INT NOT NULL DEFAULT 0,
          user_message TEXT NOT NULL,
          agent_response TEXT NOT NULL,
          routed_to TEXT,
          latency_ms INT,
          input_tokens INT,
          output_tokens INT,
          created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
        )
    """)

    cur.execute("""
        CREATE INDEX IF NOT EXISTS idx_conversations_session_id
        ON procurement.conversations (session_id, turn_index)
    """)

    conn.commit()

conn.close()
print("\u2705 OLTP tables created")

#### App Deployment

In [ ]:
import os

# Resolve the actual serving endpoint name from the tiles API.
# The MAS API auto-generates a mas-* endpoint name regardless of the endpoint_name
# field in the supervisor body, so we look it up rather than using the widget value.
SUPERVISOR_TILE_NAME = f"{CATALOG}-invoice-supervisor"
ACTUAL_SUPERVISOR_ENDPOINT = INVOICE_SUPERVISOR_ENDPOINT_NAME  # fallback
try:
    tiles_resp = w.api_client.do("GET", "/api/2.0/tiles")
    for tile in tiles_resp.get("tiles", []):
        if tile.get("name") == SUPERVISOR_TILE_NAME:
            ACTUAL_SUPERVISOR_ENDPOINT = tile.get("serving_endpoint_name", INVOICE_SUPERVISOR_ENDPOINT_NAME)
            break
    print(f"Resolved supervisor serving endpoint: {ACTUAL_SUPERVISOR_ENDPOINT}")
except Exception as e:
    print(f"Warning: tiles lookup failed ({e}), falling back to {ACTUAL_SUPERVISOR_ENDPOINT}")

app_yaml_contents = f"""command: ['npm', 'run', 'start']
env:
  - name: PGPORT
    value: '5432'
  - name: PGDATABASE
    value: '{INVOICE_DB_NAME}'
  - name: PGSSLMODE
    value: 'require'
  - name: LAKEBASE_ENDPOINT
    value: '{endpoint_name}'
  - name: INVOICE_SUPERVISOR_ENDPOINT_NAME
    value: '{ACTUAL_SUPERVISOR_ENDPOINT}'
  - name: CATALOG
    value: '{CATALOG}'
"""

app_yaml_path = os.path.abspath("../apps/invoice-terminal/app.yaml")
if os.path.exists(app_yaml_path):
    os.remove(app_yaml_path)
time.sleep(3)
with open(app_yaml_path, "w") as f:
    f.write(app_yaml_contents)
print(f"Wrote app.yaml with supervisor_endpoint={ACTUAL_SUPERVISOR_ENDPOINT}, lakebase_endpoint={endpoint_name}")

In [ ]:
from databricks.sdk.service.apps import App

source_code_path = os.path.abspath("../apps/invoice-terminal")
APP_NAME = f"{CATALOG}-invoice-terminal"

app_def = App(
    name=APP_NAME,
    default_source_code_path=source_code_path,
)

try:
    existing_app = w.apps.get(APP_NAME)
    print(f"App {APP_NAME} already exists, updating...")
    app_obj = w.apps.update(APP_NAME, app_def)
except Exception:
    app_obj = w.apps.create(app_def)

app_status = app_obj.result() if hasattr(app_obj, 'result') else app_obj
add(CATALOG, "apps", app_status)
print(f"\u2705 App {APP_NAME} ready")

#### Grants & Postgres Role

In [ ]:
import json as _json

# ── Resolve the app service principal's UUID applicationId ──────────────────
# The Databricks permissions API requires service_principal_name = UUID applicationId,
# NOT the numeric SCIM id. We use w.service_principals.get() to translate.
app_sp_application_ids = []
try:
    app_info = w.apps.get(name=APP_NAME)
    numeric_sp_id = app_info.service_principal_id  # e.g. 77381052253473

    if numeric_sp_id:
        sp_details = w.service_principals.get(numeric_sp_id)
        # application_id is the UUID used in permissions API
        app_id_uuid = sp_details.application_id
        if app_id_uuid:
            app_sp_application_ids.append(str(app_id_uuid))
            print(f"App SP applicationId (UUID): {app_id_uuid}")

    # Fallbacks in case SDK fields differ across versions
    for attr in ("service_principal_client_id", "oauth2_app_client_id"):
        val = getattr(app_info, attr, None)
        if val and str(val) not in app_sp_application_ids:
            app_sp_application_ids.append(str(val))

    print(f"Granting permissions to: {app_sp_application_ids}")
except Exception as e:
    print(f"Warning: could not resolve app SP applicationId: {e}")

app_principal_ids = app_sp_application_ids  # keep original variable name for rest of cell

if app_principal_ids:
    creds = w.postgres.generate_database_credential(endpoint=endpoint_name)
    current_user = w.current_user.me().user_name

    conn = psycopg2.connect(
        host=endpoint_host,
        port=5432,
        dbname=INVOICE_DB_NAME,
        user=current_user,
        password=creds.token,
        sslmode="require",
    )
    conn.autocommit = False

    def quote_ident(identifier):
        return '"' + identifier.replace('"', '""') + '"'

    with conn.cursor() as cur:
        cur.execute("CREATE EXTENSION IF NOT EXISTS databricks_auth")

        for principal_id in app_principal_ids:
            principal = quote_ident(principal_id)

            cur.execute("SAVEPOINT create_sp_role")
            try:
                cur.execute(
                    "SELECT databricks_create_role(%s, 'SERVICE_PRINCIPAL')",
                    (principal_id,),
                )
            except psycopg2.errors.DuplicateObject:
                cur.execute("ROLLBACK TO SAVEPOINT create_sp_role")
            except Exception as role_error:
                cur.execute("ROLLBACK TO SAVEPOINT create_sp_role")
                if "Identity not found" in str(role_error):
                    cur.execute(
                        f"DO $$ BEGIN CREATE ROLE {principal}; EXCEPTION WHEN duplicate_object THEN NULL; END $$;"
                    )
                else:
                    raise
            finally:
                cur.execute("RELEASE SAVEPOINT create_sp_role")

            cur.execute(f"GRANT USAGE, CREATE ON SCHEMA procurement TO {principal}")
            cur.execute(f"GRANT SELECT, INSERT, UPDATE, DELETE ON ALL TABLES IN SCHEMA procurement TO {principal}")
            cur.execute(f"GRANT USAGE, SELECT, UPDATE ON ALL SEQUENCES IN SCHEMA procurement TO {principal}")
            cur.execute(f"GRANT USAGE, CREATE ON SCHEMA public TO {principal}")
            cur.execute(f"GRANT SELECT, INSERT, UPDATE, DELETE ON ALL TABLES IN SCHEMA public TO {principal}")
            cur.execute(f"GRANT USAGE, SELECT, UPDATE ON ALL SEQUENCES IN SCHEMA public TO {principal}")
            cur.execute(
                f"ALTER DEFAULT PRIVILEGES IN SCHEMA procurement "
                f"GRANT SELECT, INSERT, UPDATE, DELETE ON TABLES TO {principal}"
            )
            cur.execute(
                f"ALTER DEFAULT PRIVILEGES IN SCHEMA procurement "
                f"GRANT USAGE, SELECT, UPDATE ON SEQUENCES TO {principal}"
            )
            cur.execute(
                f"ALTER DEFAULT PRIVILEGES IN SCHEMA public "
                f"GRANT SELECT, INSERT, UPDATE, DELETE ON TABLES TO {principal}"
            )
            cur.execute(
                f"ALTER DEFAULT PRIVILEGES IN SCHEMA public "
                f"GRANT USAGE, SELECT, UPDATE ON SEQUENCES TO {principal}"
            )
            print(f"Granted permissions to {principal_id}")

        conn.commit()
    conn.close()
    print("\u2705 Postgres grants applied")

    # --- Serving endpoint grants (supervisor + KA endpoints) ---
    # Resolve all endpoint names from tiles: supervisor, invoice KA, contract KA
    INVOICE_KA_TILE = f"{CATALOG}-invoice-knowledge"
    CONTRACT_KA_TILE = f"{CATALOG}-contract-knowledge"
    endpoint_names_to_grant = [ACTUAL_SUPERVISOR_ENDPOINT]

    try:
        tiles_all = w.api_client.do("GET", "/api/2.0/tiles")
        for tile in tiles_all.get("tiles", []):
            if tile.get("name") in (INVOICE_KA_TILE, CONTRACT_KA_TILE):
                ep = tile.get("serving_endpoint_name")
                if ep:
                    endpoint_names_to_grant.append(ep)
        print(f"Endpoints to grant CAN_QUERY: {endpoint_names_to_grant}")
    except Exception as e:
        print(f"Warning: tiles lookup for KA endpoints failed ({e}), only granting supervisor")

    all_eps = {ep.name: ep.id for ep in w.serving_endpoints.list()}
    for ep_name in endpoint_names_to_grant:
        ep_id = all_eps.get(ep_name)
        if not ep_id:
            print(f"Warning: endpoint {ep_name} not found, skipping")
            continue
        for principal_id in app_principal_ids:
            try:
                w.api_client.do(
                    "PATCH",
                    f"/api/2.0/permissions/serving-endpoints/{ep_id}",
                    body={
                        "access_control_list": [
                            {"service_principal_name": principal_id, "permission_level": "CAN_QUERY"}
                        ]
                    },
                )
                print(f"Granted CAN_QUERY on {ep_name} to {principal_id}")
            except Exception as e:
                print(f"Warning: could not grant CAN_QUERY on {ep_name}: {e}")

    # --- Unity Catalog grants so the app SP can read PDFs from Volumes ---
    try:
        warehouse_id = next(
            (w_obj.id for w_obj in w.warehouses.list() if w_obj.state and str(w_obj.state).endswith("RUNNING")),
            None,
        )
        if not warehouse_id:
            # fall back to any warehouse
            warehouse_id = next((w_obj.id for w_obj in w.warehouses.list()), None)

        if warehouse_id:
            for principal_id in app_principal_ids:
                sp_ref = f"`{principal_id}`"
                uc_stmts = [
                    f"GRANT USE CATALOG ON CATALOG {CATALOG} TO {sp_ref}",
                    f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.procurement TO {sp_ref}",
                    f"GRANT READ VOLUME ON VOLUME {CATALOG}.procurement.invoices TO {sp_ref}",
                    f"GRANT READ VOLUME ON VOLUME {CATALOG}.procurement.contracts TO {sp_ref}",
                ]
                for stmt in uc_stmts:
                    try:
                        result = w.statement_execution.execute_statement(
                            warehouse_id=warehouse_id,
                            statement=stmt,
                            wait_timeout="30s",
                        )
                        print(f"UC grant: {stmt[:60]}… → {result.status.state}")
                    except Exception as e:
                        print(f"Warning: UC grant failed ({stmt[:60]}…): {e}")
        else:
            print("Warning: no running warehouse found, skipping UC grants")
    except Exception as e:
        print(f"Warning: UC grants block failed: {e}")

    # --- Genie space grant (CAN_RUN) ---
    try:
        genie_df = spark.sql(f"""
            SELECT resource_data FROM {CATALOG}._internal_state.resources
            WHERE resource_type = 'genie_spaces'
            ORDER BY created_at DESC LIMIT 1
        """)
        if genie_df.count() > 0:
            genie_info = _json.loads(genie_df.first().resource_data)
            genie_space_id = genie_info.get("space_id", "")
            if genie_space_id:
                for principal_id in app_principal_ids:
                    try:
                        w.api_client.do(
                            "PATCH",
                            f"/api/2.0/permissions/genie/{genie_space_id}",
                            body={
                                "access_control_list": [
                                    {"service_principal_name": principal_id, "permission_level": "CAN_RUN"}
                                ]
                            },
                        )
                        print(f"Granted CAN_RUN on Genie space {genie_space_id} to {principal_id}")
                    except Exception as e:
                        print(f"Warning: could not grant CAN_RUN on Genie space: {e}")
        else:
            print("Warning: no Genie space found in uc_state, skipping Genie grant")
    except Exception as e:
        print(f"Warning: Genie space grant failed: {e}")
else:
    print("Warning: no app principal IDs found, skipping Postgres grants")

#### Deploy

In [ ]:
from databricks.sdk.service.apps import AppDeployment

deployment = w.apps.deploy(
    app_name=app_status.name,
    app_deployment=AppDeployment(source_code_path=source_code_path)
)
deployment_status = deployment.result() if hasattr(deployment, 'result') else deployment
print(f"App deployed: {deployment_status}")
print("\u2705 Invoice Lakebase & App stage complete")